In [ ]:
import os
import sys

# Clone your CheXFound repository into Kaggle working environment
!git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# Add CheXFound directory to system path
sys.path.append('/kaggle/working/CheXFound')

# Install dependencies if missing
!pip install -q timm einops kornia gdown

In [ ]:
import glob
import pandas as pd

# Standard paths on Kaggle for NIH dataset
possible_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
if not possible_csv:
    raise FileNotFoundError("Could not find Data_Entry_2017.csv in /kaggle/input/. Check attached datasets.")

csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# Map image filename to full file path across subfolders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# The 14 Standard Thorax Pathologies in NIH-14
CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# One-hot encode each condition
for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in x.split('|') else 0)

print(f"Total verified images indexed: {len(df)}")
df[['Image Index', 'Finding Labels'] + CLASSES[:4]].head(3)

In [ ]:
import glob

print("Searching for uploaded checkpoint files...")
for p in glob.glob('/kaggle/input/**/*', recursive=True):
    if any(k in p for k in ['teacher_checkpoint', 'glori']):
        print(f"Found: {p}")

In [ ]:
import os
import sys
import glob
import torch

sys.path.append('/kaggle/working/CheXFound')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Auto-locate the weights dynamically
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
glori_files = glob.glob('/kaggle/input/**/glori.pth', recursive=True)

assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
assert len(glori_files) > 0, "Could not find glori.pth in /kaggle/input/"

teacher_path = teacher_files[0]
glori_path = glori_files[0]

print(f"Teacher path : {teacher_path}")
print(f"GLoRI path   : {glori_path}")
print("Found teacher and glori checkpoints successfully!")

In [ ]:
!pip install -q fvcore

In [ ]:
import os

print("=== Checking /kaggle/working contents ===")
for root, dirs, files in os.walk('/kaggle/working'):
    # Only show up to 3 levels deep so it stays clean
    depth = root.replace('/kaggle/working', '').count(os.sep)
    if depth <= 2:
        print(f"{'  ' * depth}📂 {os.path.basename(root) or '/kaggle/working'}")
        py_files = [f for f in files if f.endswith('.py')][:5]
        if py_files:
            print(f"{'  ' * (depth + 1)}📄 {py_files}")

In [ ]:
import os
import sys

repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(repo_dir):
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

%cd /kaggle/working/CheXFound

# Verify chexfound folder is directly visible in current working directory
assert os.path.isdir('chexfound'), "Error: chexfound folder is not present!"
print("Path successfully registered to:", os.getcwd())

In [ ]:
import os
import sys
import importlib

# 1. Target the exact verified parent folder from your tree
target_dir = '/kaggle/working/CheXFound'
assert os.path.exists(os.path.join(target_dir, 'chexfound', '__init__.py')), "Verification failed: directory does not match tree!"

# 2. Add to sys.path at position 0
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# 3. Force Python to clear cached import failures and refresh filesystem view
importlib.invalidate_caches()
if 'chexfound' in sys.modules:
    del sys.modules['chexfound']

# 4. Import the package
import chexfound
print("SUCCESS: chexfound imported from:", chexfound.__file__)

# 5. Import submodules required for the ViT backbone and GLoRI
from chexfound.models import vision_transformer as vits
from chexfound.eval.classification.utils import setup_glori

print("SUCCESS: Vision transformer and GLoRI modules loaded without errors!")

In [ ]:
!pip install -q open_clip_torch scikit-multilearn

In [ ]:
import os
import sys
import glob
import importlib

# 1. Maintain repo path
target_dir = '/kaggle/working/CheXFound'
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

importlib.invalidate_caches()

# 2. Imports from CheXFound
import torch
import torch.nn as nn
from fvcore.common.checkpoint import Checkpointer
from chexfound.models import vision_transformer as vits
from chexfound.eval.classification.utils import setup_glori

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 3. Locate checkpoints dynamically
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
glori_files = glob.glob('/kaggle/input/**/glori.pth', recursive=True)

assert len(teacher_files) > 0, "Missing teacher_checkpoint.pth in /kaggle/input/"
assert len(glori_files) > 0, "Missing glori.pth in /kaggle/input/"

teacher_path = teacher_files[0]
glori_path = glori_files[0]
print(f"Teacher checkpoint : {teacher_path}")
print(f"GLoRI checkpoint   : {glori_path}")

# 4. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = { (k[9:] if k.startswith('backbone.') else k): v for k, v in state_dict.items() }

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
print(f"Backbone Loaded: Missing keys = {len(missing)}, Unexpected keys = {len(unexpected)}")

# Freeze foundation backbone
model.eval()
model.to(device)
for p in model.parameters():
    p.requires_grad = False

# 5. Build GLoRI Head using a dummy forward pass
dummy_input = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_output = model.get_intermediate_layers(dummy_input, n=4, return_class_token=True)

glori, _ = setup_glori(
    sample_output=sample_output,
    n_last_blocks_list=[4],
    learning_rates=[1e-4]
)

# 6. Load pretrained GLoRI weights
checkpointer = Checkpointer(glori)
checkpointer.load(glori_path)
glori.eval()
glori.to(device)

print("\nCheXFound ViT-Large backbone + GLoRI adapter successfully initialized on GPU!")

In [ ]:
#Code Base - Base line Testing using simple linear probe with 3000 sample images, and testing on 1000.
import os
import sys
import glob
import importlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Register CheXFound repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound')):
    print("Cloning CheXFound repository...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

from chexfound.models import vision_transformer as vits

# 3. Locate and load model weights
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading backbone weights from: {teacher_path}")

model = vits.vit_large(patch_size=16, img_size=512)
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = { (k[9:] if k.startswith('backbone.') else k): v for k, v in state_dict.items() }

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
model.eval()
model.to(device)
for p in model.parameters():
    p.requires_grad = False
print(f"Backbone ready on {device} (Missing: {len(missing)}, Unexpected: {len(unexpected)})")

# 4. Dataset preparation
csv_files = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(csv_files) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
df = pd.read_csv(csv_files[0])

all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)

eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_sample = train_df.sample(3000, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(1000, random_state=42).reset_index(drop=True)

class NIHFastDataset(Dataset):
    def __init__(self, data_df, transform):
        self.paths = data_df['full_path'].values
        self.labels = data_df[CLASSES].values.astype('float32')
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.labels[idx])

train_loader = DataLoader(NIHFastDataset(train_sample, eval_transform), batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(NIHFastDataset(test_sample, eval_transform), batch_size=16, shuffle=False, num_workers=2)

# 5. Linear Probe Head
class LinearProbeHead(nn.Module):
    def __init__(self, in_dim=1024, num_classes=14):
        super().__init__()
        self.classifier = nn.Linear(in_dim, num_classes)
        
    def forward(self, x):
        return self.classifier(x)

probe = LinearProbeHead(in_dim=1024, num_classes=len(CLASSES)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)

# 6. Train Linear Probe
print("\n--- Training CheXFound Linear Probe on NIH-14 ---")
for epoch in range(3):
    probe.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            tokens = model(imgs)
            if isinstance(tokens, tuple):
                tokens = tokens[0]
            if tokens.ndim == 3:
                tokens = tokens[:, 0]
                
        optimizer.zero_grad()
        logits = probe(tokens)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/3 - Loss: {total_loss / len(train_loader):.4f}")

# 7. Evaluate Baseline AUROC
probe.eval()
all_preds, all_targets = [], []

print("\nEvaluating Baseline on NIH-14 Test Split...")
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        tokens = model(imgs)
        if isinstance(tokens, tuple):
            tokens = tokens[0]
        if tokens.ndim == 3:
            tokens = tokens[:, 0]
            
        preds = torch.sigmoid(probe(tokens)).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(labels.numpy())

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# 8. Report AUROCs
aucs = []
print("\n=== Baseline CheXFound AUROC Results (NIH-14) ===")
for i, c in enumerate(CLASSES):
    if len(np.unique(all_targets[:, i])) > 1:
        score = roc_auc_score(all_targets[:, i], all_preds[:, i])
        aucs.append(score)
        print(f"{c:22s}: {score:.4f}")
    else:
        print(f"{c:22s}: N/A (no positive cases in sample)")

mean_auc = np.mean(aucs)
print("-------------------------------------------------")
print(f"Mean Baseline AUROC   : {mean_auc:.4f}")

In [ ]:
import os
import sys
import glob
import importlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Guarantee CheXFound repository existence on disk
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("CheXFound not found on disk. Cloning fresh repository...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

from chexfound.models import vision_transformer as vits

# 2. Load frozen foundation model onto GPU
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading backbone weights from: {teacher_path}")

model = vits.vit_large(patch_size=16, img_size=512)
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = { (k[9:] if k.startswith('backbone.') else k): v for k, v in state_dict.items() }

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
model.eval()
model.to(device)
for p in model.parameters():
    p.requires_grad = False
print(f"Backbone successfully loaded on GPU (Missing: {len(missing)}, Unexpected: {len(unexpected)})")

# 3. Parse NIH-14 metadata
csv_files = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(csv_files) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
df = pd.read_csv(csv_files[0])

all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)
print(f"Train Pool: {len(train_df)} | Test Pool: {len(test_df)}")

# 4. Transform & representative cohorts
eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_subset = train_df.sample(15000, random_state=42).reset_index(drop=True)
test_subset = test_df.sample(3000, random_state=42).reset_index(drop=True)

class FeatureExtractionDataset(Dataset):
    def __init__(self, data_df, transform):
        self.paths = data_df['full_path'].values
        self.labels = data_df[CLASSES].values.astype('float32')
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), torch.tensor(self.labels[idx])

train_feat_loader = DataLoader(FeatureExtractionDataset(train_subset, eval_transform), batch_size=32, shuffle=False, num_workers=2)
test_feat_loader = DataLoader(FeatureExtractionDataset(test_subset, eval_transform), batch_size=32, shuffle=False, num_workers=2)

# 5. Extract multi-block (blocks 21-24) concatenated features (4,096-dim)
def extract_features(loader, backbone, desc="Extracting"):
    all_feats, all_targets = [], []
    backbone.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc):
            imgs = imgs.to(device)
            layers = backbone.get_intermediate_layers(imgs, n=4, return_class_token=True)
            cls_tokens = torch.cat([layer[1] for layer in layers], dim=-1)
            all_feats.append(cls_tokens.cpu())
            all_targets.append(labels)
    return torch.cat(all_feats, dim=0), torch.cat(all_targets, dim=0)

print("\nPre-computing 4,096-dim representations across layers 21-24...")
X_train, y_train = extract_features(train_feat_loader, model, desc="Train Features")
X_test, y_test = extract_features(test_feat_loader, model, desc="Test Features")

print(f"\nFeature matrices ready:")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")